In [4]:
import os
from pathlib import Path
from src.TALOS.utils.common import read_yaml,create_directories
from src.TALOS.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH

In [8]:
from dataclasses import dataclass
@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir : Path
    data_path : Path

In [6]:
class ConfigurationManager:
    def __init__(self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self)->DataTransformationConfig:
        config = self.config.data_transformation
        create_directories([config.root_dir])

        return DataTransformationConfig(
            root_dir = config.root_dir,
            data_path = config.data_path
        )


In [12]:
import os
import shutil
import xml.etree.ElementTree as ET
from pathlib import Path
from src.TALOS import logger

class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        # Update this list if you have more classes in your XML
        self.classes = ["plane"]

    def convert_vbox_to_yolo(self, size, box):
        dw = 1. / size[0]
        dh = 1. / size[1]
        x = (box[0] + box[1]) / 2.0
        y = (box[2] + box[3]) / 2.0
        w = box[1] - box[0]
        h = box[3] - box[2]
        return (x * dw, y * dh, w * dw, h * dh)

    def initiate_data_transformation(self):
        try:
            source_dir = Path(self.config.data_path)
            target_root = Path(self.config.root_dir)
            split_map = {"train": "train", "test": "val"}

            for source_split, target_split in split_map.items():
                source_folder = source_dir / source_split
                img_target = target_root / "images" / target_split
                lbl_target = target_root / "labels" / target_split

                os.makedirs(img_target, exist_ok=True)
                os.makedirs(lbl_target, exist_ok=True)

                logger.info(f"Processing {source_split} split...")

                for file in os.listdir(source_folder):
                    if file.endswith(".xml"):
                        base_name = os.path.splitext(file)[0]
                        xml_path = source_folder / file
                        img_name = f"{base_name}.png"
                        img_path = source_folder / img_name
                        if os.path.exists(xml_path):
                            tree = ET.parse(xml_path)
                            root = tree.getroot()
                            size = root.find('size')
                            w = int(size.find('width').text)
                            h = int(size.find('height').text)
                            with open(lbl_target / f"{base_name}.txt", 'w') as f:
                                for obj in root.iter('object'):
                                    cls_name = obj.find('name').text
                                    if cls_name not in self.classes:
                                        continue

                                    cls_id = self.classes.index(cls_name)
                                    xmlbox = obj.find('bndbox')
                                    box = (float(xmlbox.find('xmin').text), float(xmlbox.find('xmax').text),
                                           float(xmlbox.find('ymin').text), float(xmlbox.find('ymax').text))

                                    yolo_box = self.convert_vbox_to_yolo((w, h), box)
                                    f.write(f"{cls_id} {' '.join([f'{a:.6f}' for a in yolo_box])}\n")
                        if os.path.exists(img_path):
                            shutil.copy(img_path, img_target / img_name)

            logger.info("Transformation to YOLO structure complete.")

        except Exception as e:
            raise e

In [14]:
try:
    config = ConfigurationManager()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(data_transformation_config)
    data_transformation.initiate_data_transformation()
except Exception as e:
    raise e

[2026-02-17 19:16:01,876: INFO: common: yaml_path: config/config.yaml loaded successfully]
[2026-02-17 19:16:01,878: INFO: common: yaml_path: params.yaml loaded successfully]
[2026-02-17 19:16:01,880: INFO: common: yaml_path: schema.yaml loaded successfully]
[2026-02-17 19:16:01,880: INFO: common: created directory at: artifacts]
[2026-02-17 19:16:01,880: INFO: common: created directory at: artifacts/data_transformation]
[2026-02-17 19:16:01,881: INFO: 1376285735: Processing train split...]
[2026-02-17 19:16:02,913: INFO: 1376285735: Processing test split...]
[2026-02-17 19:16:03,182: INFO: 1376285735: Transformation to YOLO structure complete.]
